<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    13/05/26  
**Docente**  Iván Carrera

# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


Para cargar el corpus primero tenemos que cargar el archivo csv.

### Carga del archivo csv


In [1]:
import pandas as pd
import os
import kagglehub

# 1. Carga del corpus (Wikipedia Text Corpus)
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")

# Buscamos el archivo CSV
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
path_csv = os.path.join(path, csv_file)

df = pd.read_csv(path_csv)

documents =df['text'].astype(str).tolist()

# IDs de los documentos
doc_ids = [f"Document_{i}" for i in range(len(documents))]

print(f"Documentos cargados: {len(documents)}")

c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documentos cargados: 10859


In [2]:
df = pd.DataFrame({'doc_id': doc_ids, 'text': documents})
df.head()

,doc_id,text
0,Document_0,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,Document_1,Battery indicator\n\nA battery indicator (also...
2,Document_2,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,Document_3,CAVNET\n\nCAVNET was a secure military forum w...
4,Document_4,CLidar\n\nThe CLidar is a scientific instrumen...


Luego de cargar el corpus nos dedicamos al preprocesamiento.

In [3]:
import nltk

# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mark_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mark_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Inicialización de herramientas
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    if not isinstance(text, str):
        return ""
    # Limpieza básica
    text_clean = re.sub(r"[^a-zA-Z0-9\s]", '', text.lower())
    # Tokenización y filtrado
    tokens = word_tokenize(text_clean)
    tokens = [t for t in tokens if t not in stop_words]
    filtered = [stemmer.stem(w) for w in tokens]
    return " ".join(filtered)

# Aplicar al dataframe
df['clean_text'] = df['text'].apply(preprocess)

Mostramos los 5 primeros documentos

In [5]:
df.head(5)

,doc_id,text,clean_text
0,Document_0,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo formerli novo comput servic compan...
1,Document_1,Battery indicator\n\nA battery indicator (also...,batteri indic batteri indic also known batteri...
2,Document_2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob peas robert allen peas august 22 1940 june...
3,Document_3,CAVNET\n\nCAVNET was a secure military forum w...,cavnet cavnet secur militari forum becam oper ...
4,Document_4,CLidar\n\nThe CLidar is a scientific instrumen...,clidar clidar scientif instrument use measur p...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

# 3. Representación Vectorial
vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.8)
start_time = time.time()
print("Calculando matriz TF-IDF para el corpus...")
tfidf_matrix = vectorizer.fit_transform(df['clean_text'])

elapsed = time.time() - start_time
print(f"✓ Matriz calculada en {elapsed:.2f} segundos")
print(f"\nDimensiones de la matriz TF-IDF:")
print(f"  → {tfidf_matrix.shape[0]} documentos  x  {tfidf_matrix.shape[1]} términos únicos")
print(f"\nTamaño en memoria (sparse): {tfidf_matrix.data.nbytes / 1024 / 1024:.2f} MB")
print(f"Elementos no cero: {tfidf_matrix.nnz:,}")

Calculando matriz TF-IDF para el corpus...
✓ Matriz calculada en 3.51 segundos

Dimensiones de la matriz TF-IDF:
  → 10859 documentos  x  5000 términos únicos

Tamaño en memoria (sparse): 14.82 MB
Elementos no cero: 1,942,808


Mostramos la matriz para los primeros 5 documentos.

In [7]:
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), 
                        columns=vectorizer.get_feature_names_out())
tfidf_df.head()

,01,05,10,100,1000,10000,100000,11,110,12,...,yield,york,young,younger,youth,youtub,zealand,zero,zinc,zone
0,0.0,0.0,0.135428,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.025707
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.0,0.000000,0.044614,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


Luego procedemos a realizar las consultas.

In [12]:
def get_snippet(text, query, window=50):
    """Busca la primera coincidencia de la query y devuelve un fragmento."""
    query_terms = query.lower().split()
    text_lower = text.lower()
    
    # Buscamos la posición del primer término de la query que aparezca
    pos = -1
    for term in query_terms:
        pos = text_lower.find(term)
        if pos != -1: break
        
    if pos == -1: return text[:window*2] + "..." # Si no hay match exacto, inicio del texto
    
    start = max(0, pos - window)
    end = min(len(text), pos + window)
    return f"...{text[start:end]}..."

In [19]:
# 4. Verificación con 10 queries
queries = ["science and technology",
            "history of ancient civilizations",
            "literature and art", 
           "mathematics and physics",
            "biological systems",
            "geography and culture",
            "political systems",
            "philosophy and ethics",
            "economics and trade",
            "nature and environment"
           ] 

def search_tfidf(query, top_n=5):
    query_clean = preprocess(query)
    query_vec = vectorizer.transform([query_clean])
    df['similarity'] = cosine_similarity( tfidf_matrix, query_vec).flatten()
    # indices = sim_scores.argsort()[-top_n:][::-1]
    # results = df.iloc[indices].copy()
    # results['similarity'] = sim_scores[indices]
    # # Añadir fragmento del texto para verificación visual
    results = df.sort_values(by='similarity', ascending=False).head(top_n).copy()
    results['snippet'] = results['text'].apply(lambda x: get_snippet(x, query))
    return results[[ 'doc_id','snippet','similarity']]

In [20]:
# Diccionario para guardar resultados si deseas compararlos después
all_results = {}

print(f"{'Query':<30} | {'Top Doc Index':<15} | {'Similitud':<10}")
print("-" * 60)

for q in queries:
    res = search_tfidf(q, top_n=3)
    all_results[q] = res
    
    # Imprimir resumen de verificación rápida
    if not res.empty:
        top_idx = res.index[0]
        top_sim = res.iloc[0]['similarity']
        print(f"{q[:30]:<30} | {top_idx:<15} | {top_sim:.4f}")

# Ejemplo: Ver detalle de la primera query
print("\nDetalle de la primera query:")
display(all_results[queries[0]])

Query                          | Top Doc Index   | Similitud 
------------------------------------------------------------
science and technology         | 618             | 0.6559
history of ancient civilizatio | 2186            | 0.3580
literature and art             | 5475            | 0.4907
mathematics and physics        | 1734            | 0.5248
biological systems             | 5592            | 0.5133
geography and culture          | 6032            | 0.7336
political systems              | 4823            | 0.2849
philosophy and ethics          | 9852            | 0.4974
economics and trade            | 2570            | 0.5277
nature and environment         | 8911            | 0.3313

Detalle de la primera query:


,doc_id,snippet,similarity
618,Document_618,...Ministry of Science and Technology (Banglad...,0.655858
3842,Document_3842,...Ministry of Science and Technology (China)\...,0.651464
1317,Document_1317,...Ministry of Science and Technology (Taiwan)...,0.618805


In [27]:
df

,doc_id,text,clean_text,similarity
0,Document_0,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo former novo comput servic compani ...,0.000000
1,Document_1,Battery indicator\n\nA battery indicator (also...,batteri indic batteri indic also known batteri...,0.000000
2,Document_2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob peas robert allen peas august 22 1940 june...,0.000000
3,Document_3,CAVNET\n\nCAVNET was a secure military forum w...,cavnet cavnet secur militari forum becam oper ...,0.000000
4,Document_4,CLidar\n\nThe CLidar is a scientific instrumen...,clidar clidar scientif instrument use measur p...,0.000000
...,...,...,...,...
10854,Document_10854,Soundcast\n\nSoundcast LLC is a privately fund...,soundcast soundcast llc privat fund compani cr...,0.000000
10855,Document_10855,Spectrum analyzer\n\nA spectrum analyzer measu...,spectrum analyz spectrum analyz measur magnitu...,0.006693
10856,Document_10856,Telepresence technology\n\nTelepresence techno...,telepres technolog telepres technolog term use...,0.078514
10857,Document_10857,Trans-Pacific Profiler Network\n\nThe Trans-Pa...,transpacif profil network transpacif profil ne...,0.000000


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 